# 3\.5\.1 Construct Social\-Integration\-Ready Mobility Datasets

This notebook builds the final mobility outlier-event handoff for the social analysis side of the project.

The purpose is simple: turn the anomaly-detection work into a clean list of mobility outlier events that can be lined up with Reddit volume, sentiment, stance, and other social signals over time.

A mobility outlier event is a place/time where the mobility pipeline found unusual behavior worth carrying forward. The event grain is:

- `taxi_zone_id`: the NYC Taxi Zone where the event happened
- `date`: the calendar date
- `temporal_bucket`: the part of day, such as weekday AM peak, weekday midday, weekday PM peak, evening, or overnight

The main output is not a full modeling table and not a full empty grid of all possible place/time combinations. It is a compact event list: only the place/time contexts where retained positive-direction mobility anomalies appeared.

The notebook exports three assets:

- `mobility_outlier_events.parquet`: the main event table for social/NLP alignment
- `mobility_outlier_event_details.parquet`: a drill-down table showing the underlying anomaly records behind each event
- `mobility_outlier_handoff_manifest.csv`: a small inventory of the exported files

Start with `mobility_outlier_events.parquet`. Use the detail table only when you need to inspect why an event was flagged.

**Read This First**

Use `mobility_outlier_events.parquet` for almost everything. It has one row per mobility outlier event and is designed to line up with the social tables by place and time.

Use `mobility_outlier_event_details.parquet` only when you need to inspect why a specific event was flagged. Do not start from the detail table unless you specifically need the underlying anomaly records.

The event table only contains detected mobility outlier events. If a social table is left-joined to the event table and the event fields are missing, that usually means there was no exported mobility outlier event for that place/time.

**Daypart Definitions**

`temporal_bucket` combines two pieces of information: whether the event falls on a weekday or weekend, and which part of day it belongs to.

The exported event table also includes `daypart`, `is_weekday_bucket`, and `is_weekend_bucket` so downstream analysis can use either the full bucket or the simpler time-of-day label.

In [1]:
# ---------------------------------------------------------------------
# Document the temporal-bucket time ranges used in the handoff
# ---------------------------------------------------------------------

import pandas as pd

temporal_bucket_definition_df = pd.DataFrame(
    [
        {
            "daypart": "am_peak",
            "time_range": "6:00am-9:59am",
            "weekday_bucket": "weekday_am_peak",
            "weekend_bucket": "weekend_am_peak",
        },
        {
            "daypart": "midday",
            "time_range": "10:00am-3:59pm",
            "weekday_bucket": "weekday_midday",
            "weekend_bucket": "weekend_midday",
        },
        {
            "daypart": "pm_peak",
            "time_range": "4:00pm-7:59pm",
            "weekday_bucket": "weekday_pm_peak",
            "weekend_bucket": "weekend_pm_peak",
        },
        {
            "daypart": "evening",
            "time_range": "8:00pm-11:59pm",
            "weekday_bucket": "weekday_evening",
            "weekend_bucket": "weekend_evening",
        },
        {
            "daypart": "overnight",
            "time_range": "12:00am-5:59am",
            "weekday_bucket": "weekday_overnight",
            "weekend_bucket": "weekend_overnight",
        },
    ]
)

display(temporal_bucket_definition_df)

,daypart,time_range,weekday_bucket,weekend_bucket
0,am_peak,6:00am-9:59am,weekday_am_peak,weekend_am_peak
1,midday,10:00am-3:59pm,weekday_midday,weekend_midday
2,pm_peak,4:00pm-7:59pm,weekday_pm_peak,weekend_pm_peak
3,evening,8:00pm-11:59pm,weekday_evening,weekend_evening
4,overnight,12:00am-5:59am,weekday_overnight,weekend_overnight


## 3\.5\.1\.1\. Define the Handoff Tables

This part defines the handoff tables before we build them.

The main table, `mobility_outlier_events.parquet`, has one row per mobility outlier event. It is designed to join cleanly with the social tables by place and time, so later analysis can compare mobility disruptions against post volume, sentiment, stance, and other social signals.

The event table keeps the fields needed for that work: location, date, daypart, policy geography, confidence, and which mobility systems were involved.

In [2]:
# ---------------------------------------------------------------------
# Document the output tables and columns before building them
# ---------------------------------------------------------------------

import pandas as pd

mobility_outlier_event_column_dictionary_df = pd.DataFrame(
    [
        {
            "column_name": "event_id",
            "table": "mobility_outlier_events",
            "meaning": "Stable event identifier for one taxi-zone/date/daypart mobility outlier event.",
        },
        {
            "column_name": "taxi_zone_id",
            "table": "mobility_outlier_events",
            "meaning": "NYC Taxi Zone ID. This is the main geography key for joining mobility events to social or neighborhood data.",
        },
        {
            "column_name": "zone",
            "table": "mobility_outlier_events",
            "meaning": "Readable Taxi Zone name.",
        },
        {
            "column_name": "borough",
            "table": "mobility_outlier_events",
            "meaning": "NYC borough for the Taxi Zone.",
        },
        {
            "column_name": "date",
            "table": "mobility_outlier_events",
            "meaning": "Calendar date of the mobility outlier event.",
        },
        {
            "column_name": "temporal_bucket",
            "table": "mobility_outlier_events",
            "meaning": "Part-of-day bucket used by the mobility pipeline, such as weekday_am_peak or weekend_evening.",
        },
        {
            "column_name": "is_weekday_bucket / is_weekend_bucket",
            "table": "mobility_outlier_events",
            "meaning": "Convenience flags split from temporal_bucket.",
        },
        {
            "column_name": "daypart",
            "table": "mobility_outlier_events",
            "meaning": "Simpler part-of-day label split from temporal_bucket, such as am_peak, midday, pm_peak, evening, or overnight.",
        },
        {
            "column_name": "pre_post_cp",
            "table": "mobility_outlier_events",
            "meaning": "Whether the event falls before or after the congestion-pricing start boundary used in this project.",
        },
        {
            "column_name": "is_pre_cp / is_post_cp",
            "table": "mobility_outlier_events",
            "meaning": "Convenience flags for pre/post congestion-pricing analysis.",
        },
        {
            "column_name": "policy_geography_class",
            "table": "mobility_outlier_events",
            "meaning": "Policy geography bucket: cbd, gateway_or_adjacent, or outside.",
        },
        {
            "column_name": "is_cbd / is_gateway_or_adjacent / is_outside",
            "table": "mobility_outlier_events",
            "meaning": "Convenience flags for policy-geography filtering.",
        },
        {
            "column_name": "is_high_confidence_outlier_event",
            "table": "mobility_outlier_events",
            "meaning": "True when at least one underlying anomaly record in this event belongs to the 2+ methods positive-direction core.",
        },
        {
            "column_name": "is_broader_positive_direction_event",
            "table": "mobility_outlier_events",
            "meaning": "True for every row in this table. The event appears in the broader retained positive-direction anomaly layer.",
        },
        {
            "column_name": "has_bus_anomaly / has_subway_anomaly / has_taxi_anomaly / has_fhvhv_anomaly / has_multimodal_anomaly",
            "table": "mobility_outlier_events",
            "meaning": "Which mobility systems contributed anomaly evidence to this event.",
        },
        {
            "column_name": "max_positive_direction_consensus_method_count",
            "table": "mobility_outlier_events",
            "meaning": "Highest number of retained anomaly methods supporting any underlying positive-direction anomaly record in this event.",
        },
        {
            "column_name": "event_id",
            "table": "mobility_outlier_event_details",
            "meaning": "Links each drill-down row back to the event table.",
        },
        {
            "column_name": "modeling_row_id",
            "table": "mobility_outlier_event_details",
            "meaning": "Original anomaly-pipeline row identifier.",
        },
        {
            "column_name": "feature_set",
            "table": "mobility_outlier_event_details",
            "meaning": "Mobility branch behind the anomaly record, such as bus, subway, taxi, fhvhv, or multimodal.",
        },
        {
            "column_name": "directional_label",
            "table": "mobility_outlier_event_details",
            "meaning": "Pipeline diagnostic label. Use only for drill-down; the main event table intentionally avoids making this the headline field.",
        },
    ]
)

export_plan_df = pd.DataFrame(
    [
        {
            "asset_name": "mobility_outlier_events",
            "asset_type": "parquet",
            "grain": "one row per taxi_zone_id x date x temporal_bucket outlier event",
            "use_this_when": "Aligning mobility outlier events with Reddit volume, sentiment, stance, or other social signals.",
        },
        {
            "asset_name": "mobility_outlier_event_details",
            "asset_type": "parquet",
            "grain": "one row per underlying anomaly record behind an event",
            "use_this_when": "Auditing or explaining why a specific event was flagged.",
        },
        {
            "asset_name": "mobility_outlier_handoff_manifest",
            "asset_type": "csv",
            "grain": "one row per exported asset",
            "use_this_when": "Checking file paths, row counts, and recommended usage.",
        },
    ]
)

display(export_plan_df)
display(mobility_outlier_event_column_dictionary_df)

,asset_name,asset_type,grain,use_this_when
0,mobility_outlier_events,parquet,one row per taxi_zone_id x date x temporal_buc...,Aligning mobility outlier events with Reddit v...
1,mobility_outlier_event_details,parquet,one row per underlying anomaly record behind a...,Auditing or explaining why a specific event wa...
2,mobility_outlier_handoff_manifest,csv,one row per exported asset,"Checking file paths, row counts, and recommend..."


,column_name,table,meaning
0,event_id,mobility_outlier_events,Stable event identifier for one taxi-zone/date...
1,taxi_zone_id,mobility_outlier_events,NYC Taxi Zone ID. This is the main geography k...
2,zone,mobility_outlier_events,Readable Taxi Zone name.
3,borough,mobility_outlier_events,NYC borough for the Taxi Zone.
4,date,mobility_outlier_events,Calendar date of the mobility outlier event.
5,temporal_bucket,mobility_outlier_events,Part-of-day bucket used by the mobility pipeli...
6,is_weekday_bucket / is_weekend_bucket,mobility_outlier_events,Convenience flags split from temporal_bucket.
7,daypart,mobility_outlier_events,Simpler part-of-day label split from temporal_...
8,pre_post_cp,mobility_outlier_events,Whether the event falls before or after the co...
9,is_pre_cp / is_post_cp,mobility_outlier_events,Convenience flags for pre/post congestion-pric...


The next block loads the final retained anomaly table from the anomaly pipeline and converts it into an event-level handoff.

The important transformation is this: the anomaly pipeline works at a more detailed row level, including separate mobility branches like bus, taxi, FHVHV, subway, and multimodal. For the social side, we collapse those rows into one mobility outlier event per `taxi_zone_id x date x temporal_bucket`.

That gives the social side one clean event record for each place/time context, while still preserving which mobility systems contributed to the signal.

In [3]:
# ---------------------------------------------------------------------
# Build the mobility outlier event and detail tables
# ---------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)

PIPELINE_DATA_DIR = Path("/datasets/_deepnote_work/pipeline_data")
FINAL_335_DIR = PIPELINE_DATA_DIR / "3.3.5.final_tables"
FINAL_351_DIR = PIPELINE_DATA_DIR / "3.5.1.final_social_integration_package"
FINAL_351_DIR.mkdir(parents=True, exist_ok=True)

RETAINED_WORKING_TABLE_PATH = FINAL_335_DIR / "retained_anomaly_working_table.parquet"
FULL_COMPARISON_TABLE_PATH = FINAL_335_DIR / "framework_comparison_full_row_universe.parquet"

OUTLIER_EVENTS_EXPORT_PATH = FINAL_351_DIR / "mobility_outlier_events.parquet"
OUTLIER_EVENT_DETAILS_EXPORT_PATH = FINAL_351_DIR / "mobility_outlier_event_details.parquet"
OUTLIER_HANDOFF_MANIFEST_PATH = FINAL_351_DIR / "mobility_outlier_handoff_manifest.csv"

WRITE_351_OUTPUTS = True

source_table_path = (
    RETAINED_WORKING_TABLE_PATH
    if RETAINED_WORKING_TABLE_PATH.exists()
    else FULL_COMPARISON_TABLE_PATH
)

source_df = pd.read_parquet(source_table_path).copy()

required_columns = [
    "modeling_row_id",
    "feature_set",
    "date",
    "temporal_bucket",
    "pre_post_cp",
    "taxi_zone_id",
    "zone",
    "borough",
    "policy_geography_class",
    "directional_label",
    "retained_positive_direction_surface_flag",
    "retained_congestion_surface_flag",
    "retained_positive_demand_surface_flag",
    "congestion_anomaly_method_count",
    "positive_demand_anomaly_method_count",
    "congestion_anomaly_two_plus_consensus_flag",
    "positive_demand_anomaly_two_plus_consensus_flag",
]
missing_columns = [col for col in required_columns if col not in source_df.columns]
assert not missing_columns, (
    "The retained anomaly package is missing required columns for the 3.5.1 handoff: "
    f"{missing_columns}"
)

flag_columns = [
    "retained_positive_direction_surface_flag",
    "retained_congestion_surface_flag",
    "retained_positive_demand_surface_flag",
    "congestion_anomaly_two_plus_consensus_flag",
    "positive_demand_anomaly_two_plus_consensus_flag",
]

for col in flag_columns:
    source_df[col] = (
        pd.to_numeric(source_df[col], errors="coerce")
        .fillna(0)
        .astype(int)
        .eq(1)
    )

source_df["date"] = pd.to_datetime(source_df["date"], errors="coerce")
source_df["taxi_zone_id"] = source_df["taxi_zone_id"].astype(str)
source_df["feature_set"] = source_df["feature_set"].astype(str).str.lower()
source_df["policy_geography_class"] = source_df["policy_geography_class"].astype(str)
source_df["pre_post_cp"] = source_df["pre_post_cp"].astype(str)
source_df["temporal_bucket"] = source_df["temporal_bucket"].astype(str)

source_df["is_high_confidence_anomaly_record"] = (
    source_df["retained_positive_direction_surface_flag"]
    & (
        (
            source_df["retained_congestion_surface_flag"]
            & source_df["congestion_anomaly_two_plus_consensus_flag"]
        )
        | (
            source_df["retained_positive_demand_surface_flag"]
            & source_df["positive_demand_anomaly_two_plus_consensus_flag"]
        )
    )
)

source_df["positive_direction_consensus_method_count"] = (
    source_df[["congestion_anomaly_method_count", "positive_demand_anomaly_method_count"]]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .max(axis=1)
    .astype(int)
)
source_df.loc[
    ~source_df["retained_positive_direction_surface_flag"],
    "positive_direction_consensus_method_count",
] = 0

outlier_detail_source_df = (
    source_df.loc[source_df["retained_positive_direction_surface_flag"]]
    .copy()
    .reset_index(drop=True)
)

outlier_detail_source_df["event_key"] = (
    outlier_detail_source_df["taxi_zone_id"].astype(str)
    + "|"
    + outlier_detail_source_df["date"].dt.strftime("%Y-%m-%d")
    + "|"
    + outlier_detail_source_df["temporal_bucket"].astype(str)
)

event_key_df = (
    outlier_detail_source_df[["event_key"]]
    .drop_duplicates()
    .sort_values("event_key")
    .reset_index(drop=True)
)
event_key_df["event_id"] = [
    f"mobility_event_{i:06d}" for i in range(1, len(event_key_df) + 1)
]

outlier_detail_source_df = outlier_detail_source_df.merge(
    event_key_df,
    on="event_key",
    how="left",
    validate="many_to_one",
)

branch_flag_df = (
    pd.crosstab(
        outlier_detail_source_df["event_key"],
        outlier_detail_source_df["feature_set"],
    )
    .gt(0)
    .reset_index()
)

for branch in ["bus", "subway", "taxi", "fhvhv", "multimodal"]:
    if branch not in branch_flag_df.columns:
        branch_flag_df[branch] = False

branch_flag_df = branch_flag_df.rename(
    columns={
        "bus": "has_bus_anomaly",
        "subway": "has_subway_anomaly",
        "taxi": "has_taxi_anomaly",
        "fhvhv": "has_fhvhv_anomaly",
        "multimodal": "has_multimodal_anomaly",
    }
)

event_base_df = (
    outlier_detail_source_df.groupby("event_key", dropna=False, observed=True)
    .agg(
        event_id=("event_id", "first"),
        taxi_zone_id=("taxi_zone_id", "first"),
        zone=("zone", "first"),
        borough=("borough", "first"),
        date=("date", "first"),
        temporal_bucket=("temporal_bucket", "first"),
        pre_post_cp=("pre_post_cp", "first"),
        policy_geography_class=("policy_geography_class", "first"),
        is_high_confidence_outlier_event=("is_high_confidence_anomaly_record", "any"),
        max_positive_direction_consensus_method_count=(
            "positive_direction_consensus_method_count",
            "max",
        ),
    )
    .reset_index()
    .merge(branch_flag_df, on="event_key", how="left", validate="one_to_one")
)

for col in [
    "has_bus_anomaly",
    "has_subway_anomaly",
    "has_taxi_anomaly",
    "has_fhvhv_anomaly",
    "has_multimodal_anomaly",
]:
    event_base_df[col] = event_base_df[col].fillna(False).astype(bool)

event_base_df["is_broader_positive_direction_event"] = True

event_base_df["is_cbd"] = event_base_df["policy_geography_class"].eq("cbd")
event_base_df["is_gateway_or_adjacent"] = event_base_df[
    "policy_geography_class"
].eq("gateway_or_adjacent")
event_base_df["is_outside"] = event_base_df["policy_geography_class"].eq("outside")
event_base_df["is_pre_cp"] = event_base_df["pre_post_cp"].eq("pre_cp")
event_base_df["is_post_cp"] = event_base_df["pre_post_cp"].eq("post_cp")
event_base_df["is_weekday_bucket"] = event_base_df["temporal_bucket"].str.startswith("weekday_")
event_base_df["is_weekend_bucket"] = event_base_df["temporal_bucket"].str.startswith("weekend_")
event_base_df["daypart"] = (
    event_base_df["temporal_bucket"]
    .str.replace("weekday_", "", regex=False)
    .str.replace("weekend_", "", regex=False)
)

mobility_outlier_events_df = event_base_df[
    [
        "event_id",
        "taxi_zone_id",
        "zone",
        "borough",
        "date",
        "temporal_bucket",
        "daypart",
        "is_weekday_bucket",
        "is_weekend_bucket",
        "pre_post_cp",
        "is_pre_cp",
        "is_post_cp",
        "policy_geography_class",
        "is_cbd",
        "is_gateway_or_adjacent",
        "is_outside",
        "is_high_confidence_outlier_event",
        "is_broader_positive_direction_event",
        "has_bus_anomaly",
        "has_subway_anomaly",
        "has_taxi_anomaly",
        "has_fhvhv_anomaly",
        "has_multimodal_anomaly",
        "max_positive_direction_consensus_method_count",
    ]
].sort_values(["date", "taxi_zone_id", "temporal_bucket"]).reset_index(drop=True)

detail_columns = [
    "event_id",
    "modeling_row_id",
    "feature_set",
    "taxi_zone_id",
    "zone",
    "borough",
    "date",
    "temporal_bucket",
    "pre_post_cp",
    "policy_geography_class",
    "directional_label",
    "is_high_confidence_anomaly_record",
    "positive_direction_consensus_method_count",
    "retained_congestion_surface_flag",
    "retained_positive_demand_surface_flag",
]

mobility_outlier_event_details_df = (
    outlier_detail_source_df[detail_columns]
    .sort_values(["event_id", "feature_set", "modeling_row_id"])
    .reset_index(drop=True)
)

outlier_event_inventory_df = pd.DataFrame(
    [
        {
            "table_name": "mobility_outlier_events",
            "rows": len(mobility_outlier_events_df),
            "columns": len(mobility_outlier_events_df.columns),
            "grain": "one row per taxi_zone_id x date x temporal_bucket mobility outlier event",
            "high_confidence_events": int(
                mobility_outlier_events_df["is_high_confidence_outlier_event"].sum()
            ),
        },
        {
            "table_name": "mobility_outlier_event_details",
            "rows": len(mobility_outlier_event_details_df),
            "columns": len(mobility_outlier_event_details_df.columns),
            "grain": "one row per anomaly record behind an exported event",
            "high_confidence_records": int(
                mobility_outlier_event_details_df["is_high_confidence_anomaly_record"].sum()
            ),
        },
    ]
)

display(outlier_event_inventory_df)
display(mobility_outlier_events_df.head(10))

,table_name,rows,columns,grain,high_confidence_events,high_confidence_records
0,mobility_outlier_events,10200,24,one row per taxi_zone_id x date x temporal_buc...,4462.0,NaN
1,mobility_outlier_event_details,10741,15,one row per anomaly record behind an exported ...,NaN,4792.0


,event_id,taxi_zone_id,zone,borough,date,temporal_bucket,daypart,is_weekday_bucket,is_weekend_bucket,pre_post_cp,is_pre_cp,is_post_cp,policy_geography_class,is_cbd,is_gateway_or_adjacent,is_outside,is_high_confidence_outlier_event,is_broader_positive_direction_event,has_bus_anomaly,has_subway_anomaly,has_taxi_anomaly,has_fhvhv_anomaly,has_multimodal_anomaly,max_positive_direction_consensus_method_count
0,mobility_event_000397,116,Hamilton Heights,Manhattan,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,outside,False,False,True,False,True,False,False,False,True,False,1
1,mobility_event_000522,129,Jackson Heights,Queens,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,outside,False,False,True,True,True,False,False,False,True,False,2
2,mobility_event_002328,136,Kingsbridge Heights,Bronx,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,outside,False,False,True,True,True,False,False,False,False,True,2
3,mobility_event_002914,142,Lincoln Square East,Manhattan,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,gateway_or_adjacent,False,True,False,True,True,False,False,True,False,False,3
4,mobility_event_003143,151,Manhattan Valley,Manhattan,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,outside,False,False,True,True,True,False,False,True,False,False,2
5,mobility_event_003242,156,Mariners Harbor,Staten Island,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,outside,False,False,True,True,True,False,False,False,False,True,2
6,mobility_event_003311,161,Midtown Center,Manhattan,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,cbd,True,False,False,False,True,False,False,False,False,True,1
7,mobility_event_003799,162,Midtown East,Manhattan,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,cbd,True,False,False,False,True,False,False,False,True,False,1
8,mobility_event_004281,174,Norwood,Bronx,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,outside,False,False,True,False,True,False,False,False,False,True,1
9,mobility_event_004555,18,Bedford Park,Bronx,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,outside,False,False,True,False,True,False,False,False,True,False,1


**What This Block Built**

The first displayed table tells you how many rows are in each handoff table.

The main event table has one row per place/time mobility outlier event. If bus, taxi, FHVHV, subway, and multimodal all flagged the same Taxi Zone on the same date and daypart, that still becomes one event row. The five `has_*_anomaly` columns show which mobility systems contributed.

The detail table keeps the underlying anomaly records for audit. Use it only when you need to explain or inspect a specific event.

The final block validates and exports the handoff. The checks are intentionally basic and practical: no missing join keys, no duplicated event IDs, one row per event context, non-empty output tables, and exported files present on disk.

In [4]:
# ---------------------------------------------------------------------
# Validate and export the mobility outlier handoff package
# ---------------------------------------------------------------------

required_event_key_columns = ["taxi_zone_id", "date", "temporal_bucket"]

validation_records = []

validation_records.extend(
    [
        {
            "check_name": "event table has no missing join keys",
            "check_value": int(mobility_outlier_events_df[required_event_key_columns].isna().any(axis=1).sum()),
            "status": "pass"
            if int(mobility_outlier_events_df[required_event_key_columns].isna().any(axis=1).sum()) == 0
            else "fail",
        },
        {
            "check_name": "event_id is unique in event table",
            "check_value": int(mobility_outlier_events_df["event_id"].duplicated().sum()),
            "status": "pass"
            if int(mobility_outlier_events_df["event_id"].duplicated().sum()) == 0
            else "fail",
        },
        {
            "check_name": "one row per taxi_zone_id/date/temporal_bucket event context",
            "check_value": int(mobility_outlier_events_df.duplicated(required_event_key_columns).sum()),
            "status": "pass"
            if int(mobility_outlier_events_df.duplicated(required_event_key_columns).sum()) == 0
            else "fail",
        },
        {
            "check_name": "event table is non-empty",
            "check_value": len(mobility_outlier_events_df),
            "status": "pass" if len(mobility_outlier_events_df) > 0 else "fail",
        },
        {
            "check_name": "detail table event IDs all appear in event table",
            "check_value": int(
                set(mobility_outlier_event_details_df["event_id"]).issubset(
                    set(mobility_outlier_events_df["event_id"])
                )
            ),
            "status": "pass"
            if set(mobility_outlier_event_details_df["event_id"]).issubset(
                set(mobility_outlier_events_df["event_id"])
            )
            else "fail",
        },
        {
            "check_name": "detail table is non-empty",
            "check_value": len(mobility_outlier_event_details_df),
            "status": "pass" if len(mobility_outlier_event_details_df) > 0 else "fail",
        },
    ]
)

outlier_handoff_validation_df = pd.DataFrame(validation_records)
display(outlier_handoff_validation_df)

assert outlier_handoff_validation_df["status"].eq("pass").all(), (
    "The mobility outlier handoff failed validation. "
    "Please inspect outlier_handoff_validation_df."
)

if WRITE_351_OUTPUTS:
    mobility_outlier_events_df.to_parquet(OUTLIER_EVENTS_EXPORT_PATH, index=False)
    mobility_outlier_event_details_df.to_parquet(
        OUTLIER_EVENT_DETAILS_EXPORT_PATH,
        index=False,
    )

mobility_outlier_handoff_manifest_df = pd.DataFrame(
    [
        {
            "asset_name": "mobility_outlier_events",
            "path": str(OUTLIER_EVENTS_EXPORT_PATH),
            "asset_type": "parquet",
            "rows": len(mobility_outlier_events_df),
            "columns": len(mobility_outlier_events_df.columns),
            "grain": "one row per taxi_zone_id x date x temporal_bucket mobility outlier event",
            "recommended_use": "Primary event table for social/NLP alignment.",
            "path_exists": OUTLIER_EVENTS_EXPORT_PATH.exists(),
        },
        {
            "asset_name": "mobility_outlier_event_details",
            "path": str(OUTLIER_EVENT_DETAILS_EXPORT_PATH),
            "asset_type": "parquet",
            "rows": len(mobility_outlier_event_details_df),
            "columns": len(mobility_outlier_event_details_df.columns),
            "grain": "one row per underlying anomaly record behind an event",
            "recommended_use": "Drill-down table for event explanation and audit.",
            "path_exists": OUTLIER_EVENT_DETAILS_EXPORT_PATH.exists(),
        },
        {
            "asset_name": "mobility_outlier_handoff_manifest",
            "path": str(OUTLIER_HANDOFF_MANIFEST_PATH),
            "asset_type": "csv",
            "rows": 3,
            "columns": 8,
            "grain": "one row per exported handoff asset",
            "recommended_use": "Small file inventory for the handoff package.",
            "path_exists": OUTLIER_HANDOFF_MANIFEST_PATH.exists(),
        },
    ]
)

if WRITE_351_OUTPUTS:
    mobility_outlier_handoff_manifest_df.to_csv(
        OUTLIER_HANDOFF_MANIFEST_PATH,
        index=False,
    )
    mobility_outlier_handoff_manifest_df["path_exists"] = mobility_outlier_handoff_manifest_df[
        "path"
    ].map(lambda p: Path(p).exists())
    mobility_outlier_handoff_manifest_df.to_csv(
        OUTLIER_HANDOFF_MANIFEST_PATH,
        index=False,
    )

export_validation_df = pd.DataFrame(
    [
        {
            "handoff_assets_listed": len(mobility_outlier_handoff_manifest_df),
            "paths_found": int(mobility_outlier_handoff_manifest_df["path_exists"].sum()),
            "status": "pass"
            if mobility_outlier_handoff_manifest_df["path_exists"].all()
            else "fail",
        }
    ]
)

display(mobility_outlier_handoff_manifest_df)
display(export_validation_df)

assert export_validation_df["status"].eq("pass").all(), (
    "One or more mobility outlier handoff exports were not written successfully."
)

,check_name,check_value,status
0,event table has no missing join keys,0,pass
1,event_id is unique in event table,0,pass
2,one row per taxi_zone_id/date/temporal_bucket ...,0,pass
3,event table is non-empty,10200,pass
4,detail table event IDs all appear in event table,1,pass
5,detail table is non-empty,10741,pass


,asset_name,path,asset_type,rows,columns,grain,recommended_use,path_exists
0,mobility_outlier_events,/datasets/_deepnote_work/pipeline_data/3.5.1.f...,parquet,10200,24,one row per taxi_zone_id x date x temporal_buc...,Primary event table for social/NLP alignment.,True
1,mobility_outlier_event_details,/datasets/_deepnote_work/pipeline_data/3.5.1.f...,parquet,10741,15,one row per underlying anomaly record behind a...,Drill-down table for event explanation and audit.,True
2,mobility_outlier_handoff_manifest,/datasets/_deepnote_work/pipeline_data/3.5.1.f...,csv,3,8,one row per exported handoff asset,Small file inventory for the handoff package.,True


,handoff_assets_listed,paths_found,status
0,3,3,pass


**Why The Event And Detail Row Counts Differ**

The event table has one row per place/time mobility outlier event. The detail table has one row per underlying anomaly record.

That is why the detail table is slightly larger. A single event can be supported by more than one mobility system, such as taxi plus multimodal. The main event table keeps one clean row for social alignment, while the detail table keeps the audit trail behind that event.

`mobility_outlier_event_details.parquet` is optional. Use it only when you need to inspect why a specific event was flagged. It includes fields that are intentionally not front-and-center in the main event table, such as:

- `modeling_row_id`: the original anomaly-pipeline row
- `feature_set`: which mobility branch produced the anomaly record
- `directional_label`: the pipeline diagnostic label for the anomaly
- `positive_direction_consensus_method_count`: how many anomaly methods supported the record
- `retained_congestion_surface_flag`
- `retained_positive_demand_surface_flag`

Most social analysis should start from `mobility_outlier_events.parquet`. The detail table is for audit, troubleshooting, or explaining a specific event.

## 3\.5\.1\.2\. Example Workflows for Using the Handoff

This section gives a few small recipes for using the exported files.

These are not meant to be final analyses. They are starting points: how to load the event table, how to filter to the higher-confidence core, how to summarize events by time or geography, and how to drill into the detail table when one event needs explanation.

In [5]:
# ---------------------------------------------------------------------
# Load the exported handoff assets for the examples below
# ---------------------------------------------------------------------

mobility_outlier_events_example_df = pd.read_parquet(OUTLIER_EVENTS_EXPORT_PATH)
mobility_outlier_event_details_example_df = pd.read_parquet(
    OUTLIER_EVENT_DETAILS_EXPORT_PATH
)

workflow_examples_df = pd.DataFrame(
    [
        {
            "example": "Start from the event table",
            "use_table": "mobility_outlier_events",
            "what_it_shows": "The main table for joining mobility outliers to social signals.",
        },
        {
            "example": "Filter to the higher-confidence core",
            "use_table": "mobility_outlier_events",
            "what_it_shows": "How many events are backed by 2+ anomaly methods.",
        },
        {
            "example": "Compare pre/post congestion-pricing periods",
            "use_table": "mobility_outlier_events",
            "what_it_shows": "A simple period summary using pre_post_cp.",
        },
        {
            "example": "Compare policy geographies",
            "use_table": "mobility_outlier_events",
            "what_it_shows": "A simple geography summary using policy_geography_class.",
        },
        {
            "example": "Check which mobility systems were involved",
            "use_table": "mobility_outlier_events",
            "what_it_shows": "How to use the five has_*_anomaly flags.",
        },
        {
            "example": "Inspect one event",
            "use_table": "mobility_outlier_event_details",
            "what_it_shows": "How to drill into the anomaly records behind one event_id.",
        },
    ]
)

display(workflow_examples_df)
display(mobility_outlier_events_example_df.head(5))

,example,use_table,what_it_shows
0,Start from the event table,mobility_outlier_events,The main table for joining mobility outliers t...
1,Filter to the higher-confidence core,mobility_outlier_events,How many events are backed by 2+ anomaly methods.
2,Compare pre/post congestion-pricing periods,mobility_outlier_events,A simple period summary using pre_post_cp.
3,Compare policy geographies,mobility_outlier_events,A simple geography summary using policy_geogra...
4,Check which mobility systems were involved,mobility_outlier_events,How to use the five has_*_anomaly flags.
5,Inspect one event,mobility_outlier_event_details,How to drill into the anomaly records behind o...


,event_id,taxi_zone_id,zone,borough,date,temporal_bucket,daypart,is_weekday_bucket,is_weekend_bucket,pre_post_cp,is_pre_cp,is_post_cp,policy_geography_class,is_cbd,is_gateway_or_adjacent,is_outside,is_high_confidence_outlier_event,is_broader_positive_direction_event,has_bus_anomaly,has_subway_anomaly,has_taxi_anomaly,has_fhvhv_anomaly,has_multimodal_anomaly,max_positive_direction_consensus_method_count
0,mobility_event_000397,116,Hamilton Heights,Manhattan,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,outside,False,False,True,False,True,False,False,False,True,False,1
1,mobility_event_000522,129,Jackson Heights,Queens,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,outside,False,False,True,True,True,False,False,False,True,False,2
2,mobility_event_002328,136,Kingsbridge Heights,Bronx,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,outside,False,False,True,True,True,False,False,False,False,True,2
3,mobility_event_002914,142,Lincoln Square East,Manhattan,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,gateway_or_adjacent,False,True,False,True,True,False,False,True,False,False,3
4,mobility_event_003143,151,Manhattan Valley,Manhattan,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,outside,False,False,True,True,True,False,False,True,False,False,2


**Example 1: Filter To The Higher-Confidence Core**

Use `is_high_confidence_outlier_event` when you want the stricter 2+ methods version of the event table. This keeps fewer events, but gives you the cleaner core we selected after the crash-validation work.

In [6]:
# ---------------------------------------------------------------------
# Example 1: filter to the higher-confidence event core
# ---------------------------------------------------------------------

higher_confidence_events_example_df = (
    mobility_outlier_events_example_df.loc[
        mobility_outlier_events_example_df["is_high_confidence_outlier_event"]
    ]
    .copy()
    .reset_index(drop=True)
)

high_confidence_event_count_df = pd.DataFrame(
    [
        {
            "all_exported_events": len(mobility_outlier_events_example_df),
            "higher_confidence_events": len(higher_confidence_events_example_df),
            "higher_confidence_event_share": (
                len(higher_confidence_events_example_df)
                / len(mobility_outlier_events_example_df)
            ),
        }
    ]
)

display(high_confidence_event_count_df.style.format({"higher_confidence_event_share": "{:.3f}"}))
display(higher_confidence_events_example_df.head(5))

,all_exported_events,higher_confidence_events,higher_confidence_event_share
0,10200,4462,0.437


,event_id,taxi_zone_id,zone,borough,date,temporal_bucket,daypart,is_weekday_bucket,is_weekend_bucket,pre_post_cp,is_pre_cp,is_post_cp,policy_geography_class,is_cbd,is_gateway_or_adjacent,is_outside,is_high_confidence_outlier_event,is_broader_positive_direction_event,has_bus_anomaly,has_subway_anomaly,has_taxi_anomaly,has_fhvhv_anomaly,has_multimodal_anomaly,max_positive_direction_consensus_method_count
0,mobility_event_000522,129,Jackson Heights,Queens,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,outside,False,False,True,True,True,False,False,False,True,False,2
1,mobility_event_002328,136,Kingsbridge Heights,Bronx,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,outside,False,False,True,True,True,False,False,False,False,True,2
2,mobility_event_002914,142,Lincoln Square East,Manhattan,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,gateway_or_adjacent,False,True,False,True,True,False,False,True,False,False,3
3,mobility_event_003143,151,Manhattan Valley,Manhattan,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,outside,False,False,True,True,True,False,False,True,False,False,2
4,mobility_event_003242,156,Mariners Harbor,Staten Island,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,outside,False,False,True,True,True,False,False,False,False,True,2


**Example 2: Compare Pre/Post Congestion-Pricing Periods**

Use `pre_post_cp` for a quick before/after summary. This is useful when the social analysis wants to compare whether outlier-event volume changes after the congestion-pricing boundary.

In [7]:
# ---------------------------------------------------------------------
# Example 2: summarize events by pre/post congestion-pricing period
# ---------------------------------------------------------------------

prepost_example_df = (
    mobility_outlier_events_example_df.groupby("pre_post_cp", dropna=False, observed=True)
    .agg(
        events=("event_id", "nunique"),
        higher_confidence_events=("is_high_confidence_outlier_event", "sum"),
    )
    .reset_index()
)
prepost_example_df["higher_confidence_event_share"] = (
    prepost_example_df["higher_confidence_events"] / prepost_example_df["events"]
)

prepost_example_df["pre_post_cp"] = pd.Categorical(
    prepost_example_df["pre_post_cp"],
    categories=["pre_cp", "post_cp"],
    ordered=True,
)
prepost_example_df = prepost_example_df.sort_values("pre_post_cp").reset_index(drop=True)

display(prepost_example_df.style.format({"higher_confidence_event_share": "{:.3f}"}))

,pre_post_cp,events,higher_confidence_events,higher_confidence_event_share
0,pre_cp,6155,2779,0.452
1,post_cp,4045,1683,0.416


**Example 3: Compare Policy Geographies**

Use `policy_geography_class` when the social analysis needs to split events into CBD, gateway/adjacent, and outside areas. The convenience flags are there too, but the class column is usually easier for grouped summaries.

In [8]:
# ---------------------------------------------------------------------
# Example 3: summarize events by policy geography
# ---------------------------------------------------------------------

policy_geo_example_df = (
    mobility_outlier_events_example_df.groupby(
        "policy_geography_class",
        dropna=False,
        observed=True,
    )
    .agg(
        events=("event_id", "nunique"),
        higher_confidence_events=("is_high_confidence_outlier_event", "sum"),
    )
    .reset_index()
)
policy_geo_example_df["higher_confidence_event_share"] = (
    policy_geo_example_df["higher_confidence_events"] / policy_geo_example_df["events"]
)

policy_geo_example_df["policy_geography_class"] = pd.Categorical(
    policy_geo_example_df["policy_geography_class"],
    categories=["cbd", "gateway_or_adjacent", "outside"],
    ordered=True,
)
policy_geo_example_df = policy_geo_example_df.sort_values(
    "policy_geography_class"
).reset_index(drop=True)

display(policy_geo_example_df.style.format({"higher_confidence_event_share": "{:.3f}"}))

,policy_geography_class,events,higher_confidence_events,higher_confidence_event_share
0,cbd,2484,904,0.364
1,gateway_or_adjacent,925,452,0.489
2,outside,6791,3106,0.457


**Example 4: See Which Mobility Systems Were Involved**

The five `has_*_anomaly` columns tell you which mobility systems contributed to an event. Use these when social analysis needs to ask whether attention lines up more with transit, taxis/FHVHV, buses, or multimodal disruption.

In [9]:
# ---------------------------------------------------------------------
# Example 4: count events involving each mobility system
# ---------------------------------------------------------------------

mobility_system_example_df = pd.DataFrame(
    [
        {
            "mobility_system": "bus",
            "events_involving_system": int(mobility_outlier_events_example_df["has_bus_anomaly"].sum()),
        },
        {
            "mobility_system": "subway",
            "events_involving_system": int(mobility_outlier_events_example_df["has_subway_anomaly"].sum()),
        },
        {
            "mobility_system": "taxi",
            "events_involving_system": int(mobility_outlier_events_example_df["has_taxi_anomaly"].sum()),
        },
        {
            "mobility_system": "fhvhv",
            "events_involving_system": int(mobility_outlier_events_example_df["has_fhvhv_anomaly"].sum()),
        },
        {
            "mobility_system": "multimodal",
            "events_involving_system": int(mobility_outlier_events_example_df["has_multimodal_anomaly"].sum()),
        },
    ]
)

mobility_system_example_df["share_of_events"] = (
    mobility_system_example_df["events_involving_system"]
    / len(mobility_outlier_events_example_df)
)

display(mobility_system_example_df.style.format({"share_of_events": "{:.3f}"}))

,mobility_system,events_involving_system,share_of_events
0,bus,1350,0.132
1,subway,428,0.042
2,taxi,1977,0.194
3,fhvhv,2189,0.215
4,multimodal,4797,0.470


**Example 5: Drill Into One Event**

When a specific event needs explanation, take its `event_id` from the event table and look it up in `mobility_outlier_event_details.parquet`. This shows the underlying anomaly records behind that one exported event.

In [10]:
# ---------------------------------------------------------------------
# Example 5: inspect the anomaly records behind one event
# ---------------------------------------------------------------------

sample_event_id = mobility_outlier_events_example_df["event_id"].iloc[0]

sample_event_df = (
    mobility_outlier_events_example_df.loc[
        mobility_outlier_events_example_df["event_id"].eq(sample_event_id)
    ]
    .copy()
    .reset_index(drop=True)
)

sample_event_detail_df = (
    mobility_outlier_event_details_example_df.loc[
        mobility_outlier_event_details_example_df["event_id"].eq(sample_event_id)
    ]
    .copy()
    .reset_index(drop=True)
)

display(sample_event_df)
display(sample_event_detail_df)

,event_id,taxi_zone_id,zone,borough,date,temporal_bucket,daypart,is_weekday_bucket,is_weekend_bucket,pre_post_cp,is_pre_cp,is_post_cp,policy_geography_class,is_cbd,is_gateway_or_adjacent,is_outside,is_high_confidence_outlier_event,is_broader_positive_direction_event,has_bus_anomaly,has_subway_anomaly,has_taxi_anomaly,has_fhvhv_anomaly,has_multimodal_anomaly,max_positive_direction_consensus_method_count
0,mobility_event_000397,116,Hamilton Heights,Manhattan,2023-01-01,weekend_overnight,overnight,False,True,pre_cp,True,False,outside,False,False,True,False,True,False,False,False,True,False,1


,event_id,modeling_row_id,feature_set,taxi_zone_id,zone,borough,date,temporal_bucket,pre_post_cp,policy_geography_class,directional_label,is_high_confidence_anomaly_record,positive_direction_consensus_method_count,retained_congestion_surface_flag,retained_positive_demand_surface_flag
0,mobility_event_000397,664163,fhvhv,116,Hamilton Heights,Manhattan,2023-01-01,weekend_overnight,pre_cp,outside,positive_demand_shock,False,1,False,True


## 3\.5\.1\.3\. Common Pitfalls and Final Handoff Notes

This final section is the plain-English usage note for the handoff.

**Use the event table as the join table.** `mobility_outlier_events.parquet` is the table to line up with social data. Join on `taxi_zone_id`, `date`, and `temporal_bucket` whenever the social table is available at the same grain.

**Use the detail table only for drill-down.** `mobility_outlier_event_details.parquet` can have more than one row for the same event because multiple mobility systems can support the same place/time outlier. That is useful for explanation, but it is not the clean join table.

**Missing event fields after a left join usually mean no exported mobility outlier event.** The event table is not a full grid of every Taxi Zone, date, and daypart. It only contains detected events.

**The higher-confidence flag is stricter.** `is_high_confidence_outlier_event` marks the event as part of the 2+ methods core. Use it when the analysis needs a cleaner, stricter signal. Keep the broader event table when coverage matters more.

**The five mobility-system flags are participation flags.** `has_bus_anomaly`, `has_subway_anomaly`, `has_taxi_anomaly`, `has_fhvhv_anomaly`, and `has_multimodal_anomaly` tell you which systems contributed to the event. They are not five separate event types.

**This handoff supports alignment, not causality by itself.** The table makes it easier to ask whether social signals changed around mobility outlier events. It does not, on its own, prove that the mobility outlier caused the social response or vice versa.

**Final recommendation:** start with `mobility_outlier_events.parquet`, filter to `is_high_confidence_outlier_event` when a stricter core is useful, and open `mobility_outlier_event_details.parquet` only when an individual event needs explanation.

**Where The Files Are**

The exported handoff package lives here:

`/datasets/_deepnote_work/pipeline_data/3.5.1.final_social_integration_package/`

Files to use:

- `mobility_outlier_events.parquet`: start here for social-table joins and most analysis
- `mobility_outlier_event_details.parquet`: use only when an individual event needs explanation or audit
- `mobility_outlier_handoff_manifest.csv`: quick inventory of exported files, row counts, and recommended usage

For most downstream work, the first file is the handoff. The other two files are there to make the handoff easier to inspect and trust.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>